## LightGCN ID Embedding V2 — 端到端图传播训练

### 配置
| 参数 | 值 | 说明 |
|------|-----|------|
| EMBED_DIM | 512 | 嵌入维度 |
| N_LAYERS | 3 | 图卷积层数 |
| EPOCHS | 30 | 训练轮数 |
| BATCH_SIZE | 2048 | 每批 (u,i) 对数 |
| LR | 1e-3 | Adam 学习率 |
| L2_REG | 0 | weight_decay=0（与 V1 相同原因） |

In [3]:
import torch, torch.nn as nn, torch.nn.functional as F
import numpy as np, pandas as pd, json, os
from scipy.sparse import csr_matrix, diags
from tqdm import tqdm
import warnings
warnings.filterwarnings("ignore")

TRAIN_CSV    = "final/train.csv"
ITEM2ID_PATH = "final/item2id.json"
OUTPUT_DIR   = "final/features"
EMBED_DIM    = 512
N_LAYERS     = 3
LR           = 1e-3
EPOCHS       = 30
BATCH_SIZE   = 2048
L2_REG       = 0
DEVICE       = "cuda" if torch.cuda.is_available() else "cpu"

os.makedirs(OUTPUT_DIR, exist_ok=True)
print(f"Device: {DEVICE}  Embed: {EMBED_DIM}d  Layers: {N_LAYERS}  Mode: end-to-end")

Device: cuda  Embed: 512d  Layers: 3  Mode: end-to-end


In [4]:
# 加载数据 & 构建归一化邻接矩阵
train = pd.read_csv(TRAIN_CSV)
train["user_id"] = train["user_id"].astype(int)
train["item_id"] = train["item_id"].astype(int)

with open(ITEM2ID_PATH) as f:
    item2id = json.load(f)
n_items = len(item2id)
n_users = train["user_id"].max() + 1
print(f"Users: {n_users}  Items: {n_items}  Interactions: {len(train)}")
print(f"Sparsity: {len(train)/(n_users*n_items)*100:.3f}%")

Users: 20030  Items: 43528  Interactions: 647950
Sparsity: 0.074%


In [5]:
user_ids = train["user_id"].values
item_ids = train["item_id"].values

N = n_users + n_items
row = np.concatenate([user_ids, item_ids + n_users])
col = np.concatenate([item_ids + n_users, user_ids])
data = np.ones(2 * len(train), dtype=np.float32)
A = csr_matrix((data, (row, col)), shape=(N, N))

degrees = np.array(A.sum(axis=1)).flatten()
deg_inv_sqrt = np.power(degrees, -0.5)
deg_inv_sqrt[np.isinf(deg_inv_sqrt)] = 0.0
D_inv_sqrt = diags(deg_inv_sqrt)
A_norm = D_inv_sqrt @ A @ D_inv_sqrt
print(f"Adjacency: {A_norm.shape}  nnz={A_norm.nnz}")

Adjacency: (63558, 63558)  nnz=1275240


In [6]:
# 初始化 E0 + 构建 PyTorch 稀疏矩阵（与 V1 相同）
E0_user = nn.Embedding(n_users, EMBED_DIM)
E0_item = nn.Embedding(n_items, EMBED_DIM)
nn.init.xavier_uniform_(E0_user.weight)
nn.init.xavier_uniform_(E0_item.weight)

def sparse_to_torch(sp_mat):
    coo = sp_mat.tocoo()
    indices = torch.LongTensor(np.vstack([coo.row, coo.col]))
    values = torch.FloatTensor(coo.data)
    return torch.sparse_coo_tensor(indices, values, sp_mat.shape)

E0_user = E0_user.to(DEVICE)
E0_item = E0_item.to(DEVICE)
A_sparse = sparse_to_torch(A_norm).to(DEVICE)

# 优化器只作用于 E0（传播层无参数）
optimizer = torch.optim.Adam(
    list(E0_user.parameters()) + list(E0_item.parameters()),
    lr=LR, weight_decay=L2_REG
)

# 正样本对 + 每用户交互物品集合（负采样用）
pos_pairs = list(zip(user_ids, item_ids))
item_set_per_user = {}
for u, i in zip(user_ids, item_ids):
    item_set_per_user.setdefault(u, set()).add(i)

print(f"Graph nodes: {N:,}  pos_pairs: {len(pos_pairs):,}  batch_size: {BATCH_SIZE}")

Graph nodes: 63,558  pos_pairs: 647,950  batch_size: 2048


### 关键差异：端到端传播 + BPR 训练

```
每 epoch:
  1. 完整图传播 (3层, 带梯度):   E0 -> A_norm x E0 -> ... -> E_final
  2. 从 E_final 采样 batch -> 算 BPR loss
  3. loss.backward() -> 梯度穿过稀疏 matmul 回传到 E0
  4. optimizer.step() 更新 E0
```

与原论文一致：E0 知道自己的最终形态会被邻居平均，朝使平均后区分度最大化的方向学习。

In [7]:
# 端到端 LightGCN 训练
# 每 epoch 先做完整图传播（带梯度），再在传播后的嵌入上算 BPR loss

print("=" * 65)
print(f"LightGCN End-to-End Training  |  {N_LAYERS}-layer  |  {EMBED_DIM}d")
print(f"Users: {n_users:,}  Items: {n_items:,}  Epochs: {EPOCHS}")
print("=" * 65)

for epoch in range(1, EPOCHS + 1):
    # 1. 完整图传播（带梯度）
    e0_all = torch.cat([E0_user.weight, E0_item.weight], dim=0)  # (N, d)
    emb_list = [e0_all]
    cur = e0_all
    for _ in range(N_LAYERS):
        cur = torch.sparse.mm(A_sparse, cur)
        emb_list.append(cur)
    # 多层均值池化
    final_all = torch.stack(emb_list, dim=0).mean(dim=0)  # (N, d)
    final_users = final_all[:n_users]                      # (n_users, d)
    final_items = final_all[n_users:]                      # (n_items, d)

    # 2. BPR 训练（在传播后的嵌入上采样 batch）
    np.random.shuffle(pos_pairs)
    total_loss, n_batch = 0, 0
    num_batches_total = (len(pos_pairs) + BATCH_SIZE - 1) // BATCH_SIZE
    pbar = tqdm(range(0, len(pos_pairs), BATCH_SIZE),
                desc=f"Epoch {epoch:2d}/{EPOCHS}", leave=True)

    for batch_idx, start in enumerate(pbar):
        batch = pos_pairs[start:start + BATCH_SIZE]
        u_ids = torch.LongTensor([p[0] for p in batch]).to(DEVICE)
        i_ids = torch.LongTensor([p[1] for p in batch]).to(DEVICE)

        # 负采样（从全部物品中随机采，排除已交互的）
        j_ids = []
        for u in u_ids.cpu().numpy():
            neg = np.random.randint(0, n_items)
            while neg in item_set_per_user.get(int(u), set()):
                neg = np.random.randint(0, n_items)
            j_ids.append(neg)
        j_ids = torch.LongTensor(j_ids).to(DEVICE)

        # 关键：从传播后的 final embedding 查表，而非 E0
        u_emb = final_users[u_ids]
        pos_emb = final_items[i_ids]
        neg_emb = final_items[j_ids]

        pos_score = (u_emb * pos_emb).sum(dim=1)
        neg_score = (u_emb * neg_emb).sum(dim=1)
        loss = -F.logsigmoid(pos_score - neg_score).mean()

        optimizer.zero_grad()
        # retain_graph=True 保留传播计算图供后续 batch 复用
        # 最后一个 batch 不 retain，自动释放
        is_last = (batch_idx == num_batches_total - 1)
        loss.backward(retain_graph=not is_last)
        optimizer.step()

        total_loss += loss.item()
        n_batch += 1
        pbar.set_postfix({"loss": f"{total_loss/n_batch:.4f}"})

    # 显式释放本轮计算图，避免显存累积
    del final_all, final_users, final_items, emb_list, e0_all, cur
    print(f"  Epoch {epoch:3d} avg_loss={total_loss/n_batch:.4f}")

print("\nTraining complete.")

LightGCN End-to-End Training  |  3-layer  |  512d
Users: 20,030  Items: 43,528  Epochs: 30


Epoch  1/30: 100%|██████████| 317/317 [00:31<00:00, 10.00it/s, loss=0.6930]


  Epoch   1 avg_loss=0.6930


Epoch  2/30: 100%|██████████| 317/317 [00:37<00:00,  8.34it/s, loss=0.6855]


  Epoch   2 avg_loss=0.6855


...

  Epoch  28 avg_loss=0.0207


Epoch 29/30: 100%|██████████| 317/317 [00:31<00:00, 10.05it/s, loss=0.0192]


  Epoch  29 avg_loss=0.0192


Epoch 30/30: 100%|██████████| 317/317 [00:31<00:00, 10.03it/s, loss=0.0178]

  Epoch  30 avg_loss=0.0178

Training complete.


In [8]:
# 提取最终物品嵌入（训练完成后最后一次传播，no_grad 节省显存）
print("Extracting final item embeddings...")
with torch.no_grad():
    e0_all = torch.cat([E0_user.weight, E0_item.weight], dim=0)
    emb_list = [e0_all]
    cur = e0_all
    for _ in range(N_LAYERS):
        cur = torch.sparse.mm(A_sparse, cur)
        emb_list.append(cur)
    final_all = torch.stack(emb_list, dim=0).mean(dim=0)
    final_items = final_all[n_users:]
    item_emb_np = final_items.cpu().numpy()

out_path = f"{OUTPUT_DIR}/item_id_lightgcn_512.npy"
np.save(out_path, item_emb_np)
print(f"Saved: {out_path}")
print(f"Shape: {item_emb_np.shape}  Min: {item_emb_np.min():.4f}  Max: {item_emb_np.max():.4f}")
print(f"Zero rows: {(item_emb_np.sum(1)==0).sum()} (expect 0)")

Extracting final item embeddings...
Saved: final/features/item_id_lightgcn_512.npy
Shape: (43528, 512)  Min: -0.7895  Max: 0.8583
Zero rows: 0 (expect 0)


In [ ]:
# 质量验证（与 V1 相同的检查）
emb = item_emb_np

# 1. NaN/Inf
print(f"NaN count:  {np.isnan(emb).sum()}")
print(f"Inf count:  {np.isinf(emb).sum()}")

# 2. 零向量
norms = np.linalg.norm(emb, axis=1)
print(f"Zero rows:  {(norms == 0).sum()}")

# 3. 范数分布
print(f"Norm:       min={norms.min():.3f} max={norms.max():.3f} mean={norms.mean():.3f} std={norms.std():.3f}")

# 4. 随机对 cosine 分布
rng = np.random.default_rng(42)
idx1 = rng.integers(0, len(emb), 5000)
idx2 = rng.integers(0, len(emb), 5000)
e1 = emb[idx1]; e2 = emb[idx2]
cos = (e1 * e2).sum(1) / (np.linalg.norm(e1, axis=1) * np.linalg.norm(e2, axis=1) + 1e-8)
print(f"Random cos: mean={cos.mean():.4f} std={cos.std():.4f} min={cos.min():.4f} max={cos.max():.4f}")

# 5. 近重复检测
high_sim = (cos > 0.95).sum()
print(f"cos>0.95:   {high_sim}/5000 ({high_sim/50:.1f}%)")

print("\nAll checks passed" if np.isnan(emb).sum() == 0 and (norms == 0).sum() == 0 else "\nIssues found")

NaN count:  0
Inf count:  0
Zero rows:  0
Norm:       min=0.857 max=6.152 mean=2.764 std=0.423
Random cos: mean=0.0274 std=0.1338 min=-0.3344 max=0.6703
cos>0.95:   0/5000 (0.0%)

All checks passed


: 

---
### V1 vs V2 对比

运行完 V2 后，将以下结果填入：

| 指标 | V1 (后处理传播) | V2 (端到端传播) |
|------|:---:|:---:|
| Epoch 1 loss | 0.6884 | ? |
| Epoch 30 loss | 0.0025 | ? |
| 最终 cosine mean | ~0 | ? |
| 域内 > 域间 | Y | ? |
| 单 epoch 耗时 | ~8s | ? |

**预期**：
- V2 初始 loss 可能与 V1 不同（传播改变了得分尺度）
- V2 收敛后 loss 可能略高于 V1（传播后的嵌入空间更平滑，极低 loss 意味着几乎完美区分，传播会引入邻居相似性）
- 推荐效果（下游 HR@10）是最终评判标准，不是 loss 数值